# Verify a GenoLeWM Receipt

This notebook verifies a checksum-only GenoLeWM receipt against its manifest. It uses the provenance surface that exists today: schema validation, manifest/model-id matching, input-commitment recomputation, and output-commitment recomputation. It does not claim model-quality assurance beyond checksum provenance.

In [1]:
from __future__ import annotations

import os
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "geno_lewm").exists():
            return candidate
    raise RuntimeError("run this notebook from inside the GenoLeWM checkout")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)

fixture_dir = ROOT / "examples" / "data" / "verify_receipt"
receipt_path = fixture_dir / "receipt.json"
manifest_path = fixture_dir / "manifest.json"
window_path = fixture_dir / "reference_window.txt"

for path in (receipt_path, manifest_path, window_path):
    print(path.relative_to(ROOT))

examples/data/verify_receipt/receipt.json
examples/data/verify_receipt/manifest.json
examples/data/verify_receipt/reference_window.txt


The fixture is intentionally small and synthetic. It is not a clinical result and is not produced by a released scorer. It exercises the same receipt and manifest contracts that a published checksum-only receipt uses.

In [2]:
from geno_lewm.provenance import load_manifest, read_receipt

manifest = load_manifest(manifest_path)
receipt = read_receipt(receipt_path)

print(f"schema={receipt.schema_version} provenance.kind={receipt.provenance.kind}")
print(f"model_id={receipt.model_id}")
print(f"manifest model_id matches receipt: {manifest.model_id() == receipt.model_id}")

schema=1.0.0 provenance.kind=checksum_only
model_id=sha256:3bcf3c87e5dd99ee9c31088ad51b9471b55999231307aeae77dc4a245363be34
manifest model_id matches receipt: True


The verifier can run with only a receipt and manifest, but passing the original reference window, edit, pooling, and dtype fields lets it recompute the input commitment too.

In [3]:
from geno_lewm.cli import verify as verify_cli

receipt_arg = str(receipt_path.relative_to(ROOT))
manifest_arg = str(manifest_path.relative_to(ROOT))
window = window_path.read_text(encoding="utf-8").strip()

rc = verify_cli.main(
    [
        receipt_arg,
        "--manifest",
        manifest_arg,
        "--input-window",
        window,
        "--edit-chrom",
        "chr17",
        "--edit-pos",
        "43091983",
        "--edit-ref",
        "A",
        "--edit-alt",
        "T",
        "--state-layer",
        "12",
        "--pool-type",
        "centered_mean",
        "--pool-radius",
        "64",
        "--normalize",
        "--encoder-dtype",
        "bf16",
        "--predictor-dtype",
        "bf16",
    ]
)
assert rc == 0

reading receipt:  examples/data/verify_receipt/receipt.json
  schema_version=1.0.0 provenance.kind=checksum_only
reading manifest: examples/data/verify_receipt/manifest.json
  model_id ok (sha256:3bcf3c87e5dd99ee\u2026)
  input_commitment ok (sha256:c482269fa95b4f7f\u2026)
  output_commitment ok (sha256:982aee9fc1786126\u2026)
ok


What this checks: the receipt is well-formed, its model id matches the manifest, the original input commitment can be recomputed, and the output block has not been altered. It does not rerun the scorer yet; that path requires the real runtime issue queue.